# Crawling Berita dari Detik.com

Lisda Lanchira Syahjian

In [ ]:
!pip install builtwith

  Preparing metadata (setup.py) ... done
  Created wheel for builtwith: filename=builtwith-1.3.4-py3-none-any.whl size=36077 sha256=bc1ae4ccf087d665eb2c7ede63d68dd21e48c4873304402c544a9534585eee57
  Stored in directory: /root/.cache/pip/wheels/7f/2d/b2/606e3df914d4aeeab99c4a4e3e9a61673d2293c2e346db00c8
Successfully built builtwith


##Scraping Berita Otomatis dari Liputan6



Kode ini digunakan untuk **mengambil judul, isi, dan kategori berita dari situs Detik.com** secara otomatis. Data yang diperoleh disimpan dalam bentuk **CSV** agar mudah diolah kembali, sekaligus ditampilkan sebagian di layar untuk memastikan hasil scraping berhasil.


In [ ]:
import builtwith

# Analisis teknologi yang digunakan
res = builtwith.parse('https://www.detik.com/')
print(res)

{'databases': ['Firebase'], 'advertising-networks': ['Google AdSense'], 'tag-managers': ['Google Tag Manager'], 'javascript-frameworks': ['jQuery']}


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

HEADERS = {"User-Agent": "Mozilla/5.0"}

# kategori yang mau diambil
KATEGORI_URLS = {
    "news": "https://news.detik.com/indeks",
    "finance": "https://finance.detik.com/indeks",
    "health": "https://health.detik.com/indeks",
    "sport": "https://sport.detik.com/indeks",
    "hot": "https://hot.detik.com/indeks"
}

def crawl_detik_indeks(max_pages=10, max_berita=150):
    data = {"id": [], "judul": [], "link": [], "kategori": [], "isi": []}
    idx = 0
    seen_links = set()

    for kategori, base_url in KATEGORI_URLS.items():
        print(f"\n=== Ambil kategori: {kategori} ===")
        for page in range(1, max_pages + 1):
            if idx >= max_berita:
                break

            url = f"{base_url}?page={page}"
            r = requests.get(url, headers=HEADERS, timeout=10)
            r.raise_for_status()
            soup = BeautifulSoup(r.text, "html.parser")

            berita_list = soup.select("h3.media__title a")

            for berita in berita_list:
                if idx >= max_berita:
                    break

                judul = berita.get_text(strip=True)
                link = berita.get("href")
                if not link.startswith("http") or link in seen_links:
                    continue
                seen_links.add(link)

                idx += 1

                # isi berita
                isi = ""
                try:
                    res = requests.get(link, headers=HEADERS, timeout=10)
                    res.raise_for_status()
                    soup_detail = BeautifulSoup(res.text, "html.parser")
                    paragraf = soup_detail.select("div.detail__body-text.itp_bodycontent p")
                    if not paragraf:
                        paragraf = soup_detail.select("div.detail__body-text p")
                    isi = " ".join(p.get_text(" ", strip=True) for p in paragraf[:5])
                except Exception:
                    isi = "(gagal ambil isi berita)"

                data["id"].append(idx)
                data["judul"].append(judul)
                data["link"].append(link)
                data["kategori"].append(kategori)
                data["isi"].append(isi)

                print(f"[{idx}] {judul} [{kategori}]")

            if idx >= max_berita:
                break

    # simpan ke CSV
    df = pd.DataFrame(data)
    df.to_csv("berita_detik_baru.csv", index=False, encoding="utf-8-sig")
    print(f"\n[DONE] {len(df)} berita unik tersimpan di berita_detik_baru.csv")
    return df


# Jalankan: ambil 150 berita dari indeks (campur kategori news, finance, health, sport, hot)
crawl_detik_indeks(max_pages=8, max_berita=150)



=== Ambil kategori: news ===
[1] Dua Tahun Berlalu, Korban Gempa Maroko Masih Hidup di Tenda Darurat [news]
[2] Alvi Pemutilasi Pacar Diamuk dan Diumpat Warga Saat Rekonstruksi di Kosan [news]
[3] Truk Seruduk 2 Angkot Lagi Ngetem di Bogor, 3 Orang Terluka [news]
[4] MK Gelar Sidang Putusan 5 Gugatan UU TNI Hari Ini [news]
[5] Gelar Razia, Pemprov Banten Temukan 86 Kendaraan ASN Nunggak Pajak [news]
[6] Bareskrim Usul Ada LO Polri di LPSK demi Perkuat Perlindungan Saksi [news]
[7] Pemobil di Pekanbaru Jadi Tersangka Usai Pukul Pejalan Kaki-Bikin Bayi Jatuh [news]
[8] Pimpinan Komisi I DPR Ungkap Djamari Chaniago Akan Jadi Menko Polkam [news]
[9] China Kumpulkan Sekutunya, Bentuk Tatanan Global Saingi AS [news]
[10] Wamentrans Sebut Pengiriman Transmigran Tergantung Permintaan Pemda [news]
[11] Ini Sosok Ken Otak Penculikan Kacab Bank demi Bobol Rekening Dormant [news]
[12] Berkas Sidang Etik 5 Anggota Brimob Pelindas Affan Masih Dilengkapi [news]
[13] Filipina vs China di Laut China S

,id,judul,link,kategori,isi
0,1,"Dua Tahun Berlalu, Korban Gempa Maroko Masih H...",https://news.detik.com/foto-news/d-8114261/dua...,news,"Maroko - Dua tahun pascagempa, ribuan warga Ma..."
1,2,Alvi Pemutilasi Pacar Diamuk dan Diumpat Warga...,https://news.detik.com/berita/d-8116286/alvi-p...,news,Rekonstruksi kasus Alvi Maulana (24) yang muti...
2,3,"Truk Seruduk 2 Angkot Lagi Ngetem di Bogor, 3 ...",https://news.detik.com/berita/d-8116283/truk-s...,news,Kecelakaan lalu lintas melibatkan truk dan dua...
3,4,MK Gelar Sidang Putusan 5 Gugatan UU TNI Hari Ini,https://news.detik.com/berita/d-8116272/mk-gel...,news,Mahkamah Konstitusi (MK) akan menggelar sidang...
4,5,"Gelar Razia, Pemprov Banten Temukan 86 Kendara...",https://news.detik.com/berita/d-8116269/gelar-...,news,Pemerintah Provinsi Banten menggelar razia ken...
...,...,...,...,...,...
145,146,Komeng Curhat ke Menhut soal Deforestasi di Ja...,https://news.detik.com/berita/d-8115487/komeng...,news,"Anggota Komite II DPD RI, Alfiansyah Komeng , ..."
146,147,Purbaya Bakal Ajak Prabowo 'Patroli' Cek Penye...,https://news.detik.com/berita/d-8115485/purbay...,news,Menteri Keuangan (Menkeu) Purbaya Yudhi Sadewa...
147,148,Kader PPP Tolak Eksternal Duduki Jabatan Ketum...,https://news.detik.com/berita/d-8115484/kader-...,news,Menjelang Muktamar Partai Persatuan Pembanguna...
148,149,Komisi V DPR-Kemendes Sepakat Bebaskan Desa-La...,https://news.detik.com/berita/d-8115483/komisi...,news,Komisi V DPR R I dan Kementerian Desa dan Pemb...
